# <Question in one line>
**Answer:** <one or two lines, written after the queries ran>

Parameters: `DAYS` (window, whole UTC days, today excluded).

In [ ]:
import os
import secrets
from databricks import sql  # pip install "databricks-sql-connector[pyarrow]" (4.x)

TOKEN = os.environ.get("DATABRICKS_TOKEN")
AUTH = {"access_token": TOKEN} if TOKEN else {"auth_type": "databricks-oauth"}
conn = sql.connect(server_hostname=os.environ["DATABRICKS_SERVER_HOSTNAME"],
                   http_path=os.environ["DATABRICKS_HTTP_PATH"], **AUTH)

def q(statement, **params):
    """Run one read-only statement with named parameters (:name) and return a DataFrame."""
    with conn.cursor() as cur:
        cur.execute(statement, params or None)
        return cur.fetchall_arrow().to_pandas()

# Keyed-hash key for masking identities (bind as :k, never print it); new per kernel.
PII_KEY = secrets.token_hex(32)

DAYS = 30

**Does:** checks that the source is visible and fresh for the window.
**Reads:** `system.billing.usage` · `usage_date` · one row · billing latency about 12 h.
**Look for:** `latest_day` = yesterday and `first_day` on or before the window start; otherwise the window is partial.

In [ ]:
q(r"""
SELECT MIN(usage_date) AS first_day, MAX(usage_date) AS latest_day, COUNT(*) AS usage_rows
FROM system.billing.usage
WHERE usage_date >= date_sub(current_date(), :days) AND usage_date < current_date()
""", days=DAYS)

**Does:** <what this query answers>.
**Reads:** <views> · <key columns> · joins on <keys> · one row = <grain> · <units>.
**Look for:** <good vs bad value>; open <row or next query> next.

In [ ]:
q(r"""
SELECT usage_date, billing_origin_product, SUM(usage_quantity) AS quantity
FROM system.billing.usage
WHERE usage_date >= date_sub(current_date(), :days) AND usage_date < current_date()
GROUP BY ALL
ORDER BY usage_date, quantity DESC
""", days=DAYS)

## Findings
- <finding, as a fact about the account, with the query that shows it>
- Not yet confirmed: <open point>

In [ ]:
# Shareable summary: counts and verdicts only, no raw identities or statement text
print("notebook: 01_<question> · window days:", DAYS)